# Power-SNR experiments
Runs the selected configs from the `powersnr-code` dataset and saves results, figures and LaTeX tables to `/kaggle/working/output.zip`.

In [ ]:
EXPERIMENTS = ['E3', 'E4', 'E5', 'E8', 'E10']   # config prefixes in configs/, e.g. "E1" -> configs/E1_*.yaml
JOBS = 4                    # Kaggle CPU sessions have 4 cores
SEEDS = None                # None = use the number of seeds in each config

In [ ]:
import glob, os, shutil, subprocess, sys
src = glob.glob('/kaggle/input/**/powersnr-code', recursive=True) or glob.glob('/kaggle/input/powersnr-code*')
src = src[0]
work = '/kaggle/working/powersnr'
if os.path.exists(work):
    shutil.rmtree(work)
shutil.copytree(src, work)
# The bundle is uploaded with --dir-mode zip; Kaggle may or may not extract the folder archives.
import zipfile
for z in glob.glob(os.path.join(work, '*.zip')):
    name = os.path.splitext(os.path.basename(z))[0]
    with zipfile.ZipFile(z) as zf:
        zf.extractall(os.path.join(work, name))
    os.remove(z)
for name in ['powersnr', 'analysis', 'configs', 'tests', 'data']:
    inner = os.path.join(work, name, name)
    if os.path.isdir(inner) and os.listdir(os.path.join(work, name)) == [name]:
        tmp = os.path.join(work, name + '__tmp')
        shutil.move(inner, tmp)
        os.rmdir(os.path.join(work, name))
        shutil.move(tmp, os.path.join(work, name))
assert os.path.exists(os.path.join(work, 'powersnr', 'runner.py')), os.listdir(work)
os.chdir(work)
sys.path.insert(0, work)
print('code from', src)
for f in sorted(glob.glob('/kaggle/input/**/*.csv*', recursive=True))[:20]:
    print(f, round(os.path.getsize(f) / 1e6, 1), 'MB')

In [ ]:
import numpy, pandas, sklearn, shap, xgboost, lightgbm
print({m.__name__: m.__version__ for m in [numpy, pandas, sklearn, shap, xgboost, lightgbm]})
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests'] if os.path.exists('tests') else ['true'])

In [ ]:
runs = []
for exp in EXPERIMENTS:
    for cfg in sorted(glob.glob(f'configs/{exp}_*.yaml')) or sorted(glob.glob(f'configs/{exp}.yaml')):
        cmd = [sys.executable, '-m', 'powersnr.runner', '--config', cfg, '--jobs', str(JOBS)]
        if SEEDS is not None:
            cmd += ['--seeds', str(SEEDS)]
        print('>>>', ' '.join(cmd), flush=True)
        subprocess.run(cmd, check=True)
        runs.append(sorted(glob.glob('results/*'))[-1])
print(runs)

In [ ]:
for run in runs:
    # assets are a convenience; a plotting failure must not lose the results
    subprocess.run([sys.executable, '-m', 'analysis.make_paper_assets', '--run', run,
                    '--out', os.path.join('paper_assets', os.path.basename(run))], check=False)
# Archive only results + assets (not the code copy).
os.makedirs('/kaggle/working/export', exist_ok=True)
for d in ['results', 'paper_assets']:
    if os.path.exists(d):
        shutil.copytree(d, f'/kaggle/working/export/{d}', dirs_exist_ok=True)
shutil.make_archive('/kaggle/working/output', 'zip', '/kaggle/working/export')
print('saved /kaggle/working/output.zip')